In [ ]:
import re
import sys
import unicodedata

START_MARKER = "# START: SAMPLE SOLUTION"
END_MARKER = "# END: SAMPLE SOLUTION"

# [ \t]*            -- Einrueckung der START-Zeile (Gruppe 1), Leerzeichen ODER Tabs
# START_MARKER       -- literaler Marker-Text
# [ \t]*\n           -- evtl. Leerzeichen am Zeilenende, dann Zeilenumbruch
# .*?                -- beliebiger Inhalt dazwischen (nicht gierig)
# \1END_MARKER        -- END-Zeile mit DERSELBEN Einrueckung wie START
# [ \t]*(\n|$)        -- evtl. Leerzeichen am Zeilenende, dann Zeilenumbruch ODER Textende
PATTERN = re.compile(
    rf"([ \t]*){re.escape(START_MARKER)}[ \t]*\n.*?\1{re.escape(END_MARKER)}[ \t]*(\n|$)",
    flags=re.DOTALL,
)

In [ ]:
def strip_source(src):
    """Entfernt alle MUSTERLOESUNG-Bloecke aus einem einzelnen Text/einer Zelle."""

    def replacement(match):
        indent = match.group(1)
        return f"{indent}raise NotImplementedError\n"

    return PATTERN.subn(replacement, src)


def check_no_leftover_markers(text, context=""):
    for marker in (START_MARKER, END_MARKER):
        if marker in text:
            raise RuntimeError(
                f"Warnung: Marker '{marker}' nach dem Entfernen noch vorhanden "
                f"{context} -- bitte Datei manuell pruefen (evtl. unpaariger Marker "
                f"oder abweichende Einrueckung zwischen START und ENDE)."
            )


def process_notebook(input_path, output_path):
    import nbformat as nbf

    nb = nbf.read(input_path, as_version=4)

    total_count = 0
    for cell in nb.cells:
        if cell.cell_type != "code":
            continue
        source = unicodedata.normalize("NFC", cell.source)
        stripped, count = strip_source(source)
        cell.source = stripped
        total_count += count

    print(f"{total_count} Musterloesungs-Block(-e) entfernt (Notebook).")

    # Sicherheitscheck ueber alle Zellen hinweg
    combined = "\n".join(c.source for c in nb.cells)
    check_no_leftover_markers(combined)

    nbf.validate(nb)

    with open(output_path, "w", encoding="utf-8") as f:
        nbf.write(nb, f)

In [ ]:
for i in [1, 2, 3]:

    input_path = f"/home/jovyan/mri/solutions/exams-solution/mock-exam-{i}-solution.ipynb"
    output_path = f"/home/jovyan/mri/exercises/exams/mock-exam-{i}.ipynb"

    process_notebook(input_path, output_path)
    
    print(f"Klausurversion geschrieben nach: {output_path}")